In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / "data"

orders = pd.read_csv(DATA / "olist_orders_dataset.csv")
items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA / "olist_order_payments_dataset.csv")

In [2]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [3]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [4]:
items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [5]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [6]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [7]:
payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [8]:
items["item_total"] = items["price"] + items["freight_value"]

print(items["item_total"].sum())
print(payments["payment_value"].sum())

15843553.24
16008872.12


In [9]:
payment_summary = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
)

item_summary = (
    items
    .groupby("order_id")["item_total"]
    .sum()
    .reset_index()
)

comparison = payment_summary.merge(
    item_summary,
    on="order_id",
    how="inner"
)

comparison["difference"] = (
    comparison["payment_value"]
    - comparison["item_total"]
)

comparison.head()

,order_id,payment_value,item_total,difference
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,72.19,0.000000e+00
1,00018f77f2f0320c557190d7a144bdd3,259.83,259.83,0.000000e+00
2,000229ec398224ef6ca0657da4fc703e,216.87,216.87,0.000000e+00
3,00024acbcdf0a6daa1e931b038114c75,25.78,25.78,0.000000e+00
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,218.04,-2.842171e-14


In [10]:
comparison["difference"].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: difference, dtype: float64

In [15]:
real_diff = comparison[
    comparison["difference"].abs() > 0.01
]
print(real_diff.shape)

real_diff.head(20)

(378, 4)


,order_id,payment_value,item_total,difference
165,00789ce015e7e5791c7914f32bb4fad4,190.81,168.83,21.98
525,016726239765c18f66826453f39c64e3,265.77,235.13,30.64
724,01e51b7c3025655646143d09b911e1d7,35.02,33.10,1.92
965,028aa7c930356788f861ed1b7f984819,62.94,57.53,5.41
1123,02f4dd90ba0feb8ec394cac05862d2b5,141.65,130.96,10.69
1233,033ccfbdfc4d29677b7e1e6df3a82820,69.26,59.96,9.30
1248,0345ba423eed2b009e0e407b19c422e4,83.17,77.75,5.42
1418,03b218d39c422c250f389120c531b61f,58.03,50.24,7.79
1789,04993613aee4046caf92ea17b316dcfb,524.28,524.32,-0.04
1887,04e00ba23c33890eaee39b02e8185cc2,53.41,48.30,5.11


A small proportion (0.38%) of orders showed discrepancies between payment values and item totals. Since payment_value represents the final customer payment, it was selected as the official revenue metric.